# Test vLLM-Omni Docker Serving Engine

This notebook demonstrates how to test the running `vLLM-Omni` server using **LiteLLM** and standard HTTP requests.

- Multimodal Chat (Vision / Audio / Text)
- Text-to-Image Diffusion (DiT models)
- GPU Memory Monitoring (`nvidia-smi`)

In [ ]:
# Check current GPU status & VRAM utilization
!nvidia-smi

In [ ]:
import base64
import json
import os
import requests

# Check LiteLLM availability
try:
    import litellm
    print("[+] LiteLLM is installed and available (version:", getattr(litellm, '__version__', 'unknown'), ")")
except ImportError:
    print("[-] LiteLLM is not installed. To install: pip install litellm")
    print("    (Notebook will fall back to direct requests where needed)")

In [ ]:
# 1. Health check & Query Active Models from the vLLM-Omni server
API_BASE = "http://localhost:8091/v1"

try:
    resp = requests.get(f"{API_BASE}/models", timeout=10)
    resp.raise_for_status()
    models_data = resp.json()
    active_models = [m["id"] for m in models_data.get("data", [])]
    print("[+] Server is running at", API_BASE)
    print("[+] Available models:", active_models)
except Exception as e:
    print("[!] Failed to connect to vLLM-Omni server:", e)
    print("    Make sure the docker container is running (e.g. ./src/run-vllm-omni-docker.sh)")
    active_models = []

## 2. Multimodal Chat Test (Vision / Audio / Text)

Testing `/v1/chat/completions` with an image URL using LiteLLM (or requests fallback).

In [ ]:
# Determine active model name
model_name = active_models[0] if active_models else "qwen3-omni"
print(f"Target model: {model_name}")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Describe what you see in this picture in 20 words:"},
            {
                "type": "image_url",
                "image_url": {
                    "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg"
                },
            },
        ],
    }
]

try:
    import litellm
    print("[*] Calling via LiteLLM...")
    response = litellm.completion(
        model=f"openai/{model_name}",
        api_base=API_BASE,
        api_key="none",
        messages=messages,
        temperature=0.7,
    )
    reply = response.choices[0].message.content
    print("\n[+] Response from LiteLLM:\n", reply)
except Exception as e:
    print("[*] LiteLLM call skipped/failed, using direct HTTP request:", e)
    try:
        resp = requests.post(
            f"{API_BASE}/chat/completions",
            json={"model": model_name, "messages": messages},
            timeout=60,
        )
        resp.raise_for_status()
        reply = resp.json()["choices"][0]["message"]["content"]
        print("\n[+] Response from Requests:\n", reply)
    except Exception as http_err:
        print("[-] HTTP Request error:", http_err)

## 3. Diffusion / Text-to-Image Generation

Testing `/v1/images/generations` with LiteLLM (`image_generation`).

In [ ]:
prompt = "A cute red panda wearing a tiny astronaut helmet on Mars, cinematic photography, high detail"
output_file = "omni_generated_image.png"
image_model = active_models[0] if active_models else "qwen-image"

print(f"Target model: {image_model}")
print(f"Prompt:       {prompt}")

try:
    import litellm
    print("[*] Calling litellm.image_generation...")
    img_resp = litellm.image_generation(
        model=f"openai/{image_model}",
        api_base=API_BASE,
        api_key="none",
        prompt=prompt,
        size="1024x1024",
    )
    item = img_resp.data[0]
    if hasattr(item, "b64_json") and item.b64_json:
        with open(output_file, "wb") as f:
            f.write(base64.b64decode(item.b64_json))
        print(f"[+] Image generated and saved to {output_file}")
    elif hasattr(item, "url") and item.url:
        print("[+] Image URL:", item.url)
except Exception as e:
    print("[*] LiteLLM image call skipped/failed, using direct HTTP request:", e)
    try:
        resp = requests.post(
            f"{API_BASE}/images/generations",
            json={"model": image_model, "prompt": prompt, "size": "1024x1024"},
            timeout=120,
        )
        resp.raise_for_status()
        item = resp.json()["data"][0]
        if "b64_json" in item:
            with open(output_file, "wb") as f:
                f.write(base64.b64decode(item["b64_json"]))
            print(f"[+] Image generated and saved to {output_file}")
        elif "url" in item:
            print("[+] Image URL:", item["url"])
    except Exception as http_err:
        print("[-] HTTP Image generation error:", http_err)

## 4. Display Generated Image

In [ ]:
from IPython.display import Image, display

if os.path.exists(output_file):
    display(Image(filename=output_file))
else:
    print("Image file not yet created. Run the generation cell above once a diffusion server is running.")

## 5. Post-Run GPU VRAM Monitoring

In [ ]:
!nvidia-smi